In [ ]:
print("="*70)
print("ArbItro - Pipeline 2 Test")
print("="*70)

import tensorflow as tf
print(f"\nTensorFlow version: {tf.__version__}")

gpu_name = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if gpu_name:
    gpu_info = gpu_name[0].split(',')
    print(f"  GPU  : {gpu_info[0].strip()}")
    print(f"  VRAM : {gpu_info[1].strip()}")
    print("\n  System Resources:")
    !free -h | grep Mem
    !nvidia-smi --query-gpu=memory.free,memory.total --format=csv
else:
    print("\n  No GPU found")

print("\n" + "="*70)

In [ ]:
import os, sys

try:
    from google.colab import drive
    USE_COLAB = True
    drive.mount('/content/drive')
    BASE_DIR      = '/content/drive/MyDrive/ArbItro'
    CODE_DIR      = os.path.join(BASE_DIR, 'ArbItro_code')
    DATASET_DIR   = os.path.join(BASE_DIR, 'ArbItro_dataset', 'dataset')
    TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')
except ImportError:
    USE_COLAB = False
    BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
    CODE_DIR      = os.path.join(BASE_DIR, 'model', 'src', 'pipeline2')
    DATASET_DIR   = os.path.join(BASE_DIR, 'dataset')
    TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')

sys.path.insert(0, CODE_DIR)

print(f"  USE_COLAB    : {USE_COLAB}")
print(f"  CODE_DIR     : {CODE_DIR}")
print(f"  DATASET_DIR  : {DATASET_DIR}")
print(f"  TRAINING_ROOT: {TRAINING_ROOT}")

required_paths = [
    os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    os.path.join(DATASET_DIR, 'test'),
]
print("\nVerifying test paths:")
for path in required_paths:
    status = "Found" if os.path.exists(path) else "Not found"
    print(f"  {status}: {path}")

In [ ]:
!pip install opencv-python-headless

In [ ]:
if USE_COLAB:
    !mkdir -p /content/arbitro_model
    %cd /content/arbitro_model
    !cp "{CODE_DIR}/data_loader.py" .
    !cp "{CODE_DIR}/model.py" .
    sys.path.insert(0, '/content/arbitro_model')
    print("\nFiles copied to /content/arbitro_model:")
    !ls -lh

In [ ]:
try:
    from data_loader import ArbItroDataGenerator
    from model import (
        build_arbitro_model_speed_aware_lstm_multiclip,
        compile_arbitro_model,
        BinaryBalancedAccuracy,
        MulticlassBalancedAccuracy,
    )
    print("  Pipeline 2 imports successful!")
except Exception as e:
    print(f"  Import error: {e}")
    raise

In [ ]:
TEST_CONFIG = {
    'json_path'  : os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    'video_dir'  : os.path.join(DATASET_DIR, 'test'),
    'model_path' : os.path.join(TRAINING_ROOT, 'models', 'pipeline2.keras'),
    'dim'        : (224, 398),
    'n_frames'   : 16,
    'max_clips'  : 4,
    'batch_size' : 28,
    'shuffle'    : False,
}

_OFF_MAP = {"No offence": 0, "": 0, "Between": 1, "Offence": 1}
_ACT_MAP = {
    "": 0, "Dont know": 0, "Dive": 0,
    "Challenge": 1, "Tackling": 1, "Standing tackling": 1,
    "Holding": 2, "Pushing": 2, "Elbowing": 2,
    "High leg": 3,
}
MAP_OFF = {0: "No Offence",   1: "Offence"}
MAP_SEV = {0: "No Card",      1: "Yellow",      2: "Red"}
MAP_ACT = {0: "Neutral/Dive", 1: "Tackles",     2: "Upper Body", 3: "High Leg"}

def _get_sev(row):
    try:
        v = float(row.get('Severity', 0))
        if v >= 5.0: return 2
        if v >= 3.0: return 1
        return 0
    except: return 0

status = "Found" if os.path.exists(TEST_CONFIG['model_path']) else "Not found"
print(f"  {status}: {TEST_CONFIG['model_path']}")

In [ ]:
import traceback

try:
    test_gen = ArbItroDataGenerator(
        json_path=TEST_CONFIG['json_path'],
        base_video_path=TEST_CONFIG['video_dir'],
        batch_size=TEST_CONFIG['batch_size'],
        max_clips=TEST_CONFIG['max_clips'],
        dim=TEST_CONFIG['dim'],
        n_frames=TEST_CONFIG['n_frames'],
        shuffle=False,
        use_auxiliary_features=True,
        augment=False,
    )
    print(f"  Test samples : {test_gen.n_samples}")
    print(f"  Test batches : {len(test_gen)}")
except Exception as e:
    print(f"  Generator error: {e}")
    traceback.print_exc()
    raise


def masked_mean(inputs):
    x, mask = inputs[0], inputs[1]
    mask = tf.expand_dims(mask, axis=-1)
    return tf.reduce_sum(x * mask, axis=1) / (tf.reduce_sum(mask, axis=1) + 1e-7)


try:
    model = tf.keras.models.load_model(
        TEST_CONFIG['model_path'],
        custom_objects={'masked_mean': masked_mean},
        compile=False,
    )
    print(f"\n  Model loaded : {TEST_CONFIG['model_path']}")
    print(f"  Output heads : {model.output_names}")
except Exception as e:
    print(f"  Error loading model: {e}")
    traceback.print_exc()
    raise

In [ ]:
import numpy as np

print("Running inference (Pipeline 2)...")
predictions = model.predict(test_gen, verbose=1)
pred_dict = {name: val for name, val in zip(model.output_names, predictions)}
print(f"  Output keys: {list(pred_dict.keys())}")

In [ ]:
import json
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix as sk_cm


def print_raw_report(y_true, y_pred, labels, title):
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    print(f"  Accuracy         : {acc:.2%}")
    print(f"  Balanced Accuracy: {bal_acc:.2%}")
    print("-" * 70)
    cm = sk_cm(y_true, y_pred, labels=list(labels.keys()))
    print(f"  {'CLASS':<22} | {'CORRECT / TOTAL':<18} | {'CLASS ACC':<10}")
    print("-" * 70)
    for idx, name in labels.items():
        if idx < len(cm):
            total = int(np.sum(cm[idx, :]))
            correct = int(cm[idx, idx])
            acc_c = correct / total if total > 0 else 0.0
            print(f"  {name:<22} | {correct}/{total:<17} | {acc_c:.2%}")
    print("-" * 70)
    return {'accuracy': float(acc), 'balanced_accuracy': float(bal_acc)}


def print_action_raw_report(test_samples, y_pred, raw_classes, act_map, title):
    y_true = np.array([act_map.get(s.get('Action class', ''), 0) for s in test_samples])
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)
    print(f"  Accuracy         : {acc:.2%}")
    print(f"  Balanced Accuracy: {bal_acc:.2%}")
    print("-" * 70)
    print(f"  {'CLASS':<22} | {'CORRECT / TOTAL':<18} | {'CLASS ACC':<10}")
    print("-" * 70)
    for cls_name in raw_classes:
        indices = [i for i, s in enumerate(test_samples) if s.get('Action class', '') == cls_name]
        target_grp = act_map.get(cls_name, 0)
        correct = sum(1 for i in indices if y_pred[i] == target_grp)
        total = len(indices)
        if total > 0:
            label = "N/A (Empty)" if cls_name == "" else cls_name
            print(f"  {label:<22} | {correct}/{total:<17} | {correct / total:.2%}")
    print("-" * 70)
    return {'accuracy': float(acc), 'balanced_accuracy': float(bal_acc)}


num_samples = pred_dict[next(iter(pred_dict))].shape[0]
test_samples = test_gen.samples[:num_samples]

y_true_off = np.array([_OFF_MAP.get(s.get('Offence', ''), 0) for s in test_samples])
y_pred_off = (pred_dict['head_offence'].flatten() > 0.5).astype(int)
res_off = print_raw_report(y_true_off, y_pred_off, MAP_OFF, "HEAD OFFENCE (Pipeline 2)")

y_pred_act = np.argmax(pred_dict['head_action'], axis=1)
raw_act_classes = ["", "Dont know", "Challenge", "Tackling", "Standing tackling",
                   "High leg", "Holding", "Pushing", "Elbowing", "Dive"]

res_act = print_action_raw_report(test_samples, y_pred_act, raw_act_classes, _ACT_MAP,
                                  "HEAD ACTION (Pipeline 2 — Raw Classes)")

y_true_sev = np.array([_get_sev(s) for s in test_samples])
y_pred_sev = np.argmax(pred_dict['head_severity'], axis=1)
res_sev = print_raw_report(y_true_sev, y_pred_sev, MAP_SEV, "HEAD SEVERITY (Pipeline 2)")

os.makedirs(os.path.join(TRAINING_ROOT, 'models'), exist_ok=True)
results_path = os.path.join(TRAINING_ROOT, 'models', 'test_results_pipeline2.json')
with open(results_path, 'w') as f:
    json.dump({'offence': res_off, 'action': res_act, 'severity': res_sev}, f, indent=2)
print(f"\n  Results saved: {results_path}")